# Semana 9 — MDP de horizonte infinito: Programación Lineal (LP)

`jmarkov` sigue siendo la **herramienta principal** del curso para resolver MDP de horizonte infinito — pero `dtmdp` internamente solo sabe resolver con VI o con PI, **no** trae un método de Programación Lineal. Por eso, para el tema de esta semana, `jmarkov` cumple un papel distinto al de siempre: en vez de ser la herramienta que resuelve, es la **referencia** contra la cual construimos y validamos la formulación de LP. La herramienta que sí resuelve el LP es `PuLP`.

En cada ejercicio seguimos entonces este orden:

1. Planteamos el MDP (estados, acciones, matrices, retornos) — igual que siempre.
2. **Resolvemos primero con `jmarkov`** (`dtmdp`), para tener de inmediato el valor óptimo de referencia, sin tener que esperar a terminar de formular el LP.
3. Formulamos el mismo problema como programa lineal y lo resolvemos con `PuLP` — esta es la técnica nueva de la semana.
4. Confirmamos que el resultado de `PuLP` reproduce el valor de referencia que ya teníamos desde el paso 2.

## La formulación de LP

$$
\min_V \sum_s V_s \qquad \text{sujeto a} \qquad V_s \geq r(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, V_{s'} \qquad \forall (s,a).
$$

¿Por qué "la más pequeña" función $V$ que satisface las desigualdades? Porque cualquier $V$ que las satisfaga es una cota superior de $V^*$; minimizar $\sum_s V_s$ obliga a que las desigualdades se vuelvan igualdades exactamente donde importa, y eso es la ecuación de Bellman de optimalidad. Hay una restricción por cada par $(s,a)$: $|\mathcal S|\cdot|\mathcal A|$ en total.

### Herramienta nueva: `PuLP`

```python
import pulp

problema = pulp.LpProblem("nombre", pulp.LpMinimize)   # 1. crear el problema
V = [pulp.LpVariable(f"V_{s}", lowBound=None) for s in range(n)]  # 2. variables (libres: V puede ser negativo)
problema += pulp.lpSum(V)                                 # 3. funcion objetivo
problema += V[0] >= 5.2                                   # 4. cada restriccion se agrega con +=
problema.solve(pulp.PULP_CBC_CMD(msg=0))                  # 5. resolver con CBC
pulp.value(V[0])                                           # 6. leer el valor de una variable
```

`pulp.LpStatus[problema.status]` dice si el problema terminó en `"Optimal"` — conviene revisarlo siempre antes de confiar en los valores.

In [1]:
import numpy as np
import time
import pulp

from jmarkov.mdp.dtmdp import dtmdp

np.set_printoptions(precision=4, suppress=True)
print("Version de PuLP:", pulp.__version__)

Version de PuLP: 3.3.2


## Herramienta: `np.argmax`

Una vez resuelto el LP tenemos $V^*$, pero no la política: la recuperamos recalculando el valor de cada acción con $V^*$ y tomando el `argmax` — igual que hicimos con VI y PI en la Semana 8.

In [2]:
valores_ejemplo = np.array([4.2, 6.8])
print("Mejor valor:", np.max(valores_ejemplo))
print("Índice de la mejor acción:", np.argmax(valores_ejemplo))

Mejor valor: 6.8
Índice de la mejor acción: 1


---
# Ejercicio 1 — Gestión de activos: reemplazo de un generador

Mismo ejemplo del cuaderno de la Semana 8.

$$
\mathcal S=\{\text{Nuevo},\text{Bueno},\text{Regular},\text{Malo}\}, \qquad \mathcal A=\{\text{Mantener},\text{Reemplazar}\}.
$$

$$
P^{(M)}=\begin{pmatrix}0.85&0.15&0&0\\0&0.80&0.20&0\\0&0&0.70&0.30\\0&0&0&1\end{pmatrix}, \qquad P^{(R)}=\begin{pmatrix}1&0&0&0\\0.85&0.15&0&0\\0.85&0.15&0&0\\0.85&0.15&0&0\end{pmatrix}.
$$

| Estado | Mantener | Reemplazar |
|:---|---:|---:|
| Nuevo | 100 | -10000 |
| Bueno | 80 | -20 |
| Regular | 50 | -20 |
| Malo | 20 | -20 |

Factor de descuento: $\gamma=0.90$.

In [3]:
estados_generador = np.array(["Nuevo", "Bueno", "Regular", "Malo"])
acciones_generador = np.array(["Mantener", "Reemplazar"])
gamma_generador = 0.90

P_mantener = np.array([
    [0.85, 0.15, 0.00, 0.00],
    [0.00, 0.80, 0.20, 0.00],
    [0.00, 0.00, 0.70, 0.30],
    [0.00, 0.00, 0.00, 1.00]
])

P_reemplazar = np.array([
    [1.00, 0.00, 0.00, 0.00],
    [0.85, 0.15, 0.00, 0.00],
    [0.85, 0.15, 0.00, 0.00],
    [0.85, 0.15, 0.00, 0.00]
])

matrices_generador = [P_mantener, P_reemplazar]

retornos_generador = np.array([
    [100.0, -10000.0],
    [ 80.0,    -20.0],
    [ 50.0,    -20.0],
    [ 20.0,    -20.0]
])

numero_estados_generador = len(estados_generador)
numero_acciones_generador = len(acciones_generador)

for indice_accion in range(numero_acciones_generador):
    print("Accion:", acciones_generador[indice_accion])
    for estado_actual in range(numero_estados_generador):
        suma_fila = matrices_generador[indice_accion][estado_actual, :].sum()
        print("  Estado", estados_generador[estado_actual], "- suma de la fila:", round(float(suma_fila), 6))

Accion: Mantener
  Estado Nuevo - suma de la fila: 1.0
  Estado Bueno - suma de la fila: 1.0
  Estado Regular - suma de la fila: 1.0
  Estado Malo - suma de la fila: 1.0
Accion: Reemplazar
  Estado Nuevo - suma de la fila: 1.0
  Estado Bueno - suma de la fila: 1.0
  Estado Regular - suma de la fila: 1.0
  Estado Malo - suma de la fila: 1.0


## 1.1 Primero, la referencia con `jmarkov`

Antes de construir el LP, resolvemos el MDP con `jmarkov` (que ya conocemos de la Semana 8) para tener de inmediato el valor óptimo de referencia.

In [4]:
matrices_jmarkov_generador = {
    "Mantener": matrices_generador[0],
    "Reemplazar": matrices_generador[1]
}

mdp_generador = dtmdp(
    estados_generador, acciones_generador, matrices_jmarkov_generador, retornos_generador, gamma_generador
)

valor_referencia_generador, politica_referencia_generador = mdp_generador.solve(1e-10, minimize=False)

print("Valor de referencia (jmarkov):")
for estado_actual in range(numero_estados_generador):
    print(" ", estados_generador[estado_actual], ":", round(float(valor_referencia_generador[estado_actual]), 4))

print()
print("Politica de referencia (jmarkov):", politica_referencia_generador)

Valor de referencia (jmarkov):
  Nuevo : 864.6747
  Bueno : 764.4337
  Regular : 744.6747
  Malo : 744.6747

Politica de referencia (jmarkov): {np.str_('Nuevo'): np.str_('Mantener'), np.str_('Bueno'): np.str_('Mantener'), np.str_('Regular'): np.str_('Reemplazar'), np.str_('Malo'): np.str_('Reemplazar')}


## 1.2 Ahora, la formulación de LP con `PuLP`

Traducimos la formulación de la introducción directamente a `PuLP`:

1. Una variable $V_s$ por estado, **sin cota inferior**.
2. Función objetivo: $\min \sum_s V_s$.
3. Una restricción por cada par $(s,a)$.

In [5]:
problema_generador = pulp.LpProblem("MDP_generador", pulp.LpMinimize)

V_generador = []
for estado_actual in range(numero_estados_generador):
    V_generador.append(pulp.LpVariable(f"V_{estado_actual}", lowBound=None))

problema_generador += pulp.lpSum(V_generador)

for estado_actual in range(numero_estados_generador):
    for accion_actual in range(numero_acciones_generador):

        continuacion = pulp.lpSum(
            matrices_generador[accion_actual][estado_actual, estado_siguiente] * V_generador[estado_siguiente]
            for estado_siguiente in range(numero_estados_generador)
        )

        problema_generador += V_generador[estado_actual] >= retornos_generador[estado_actual, accion_actual] + gamma_generador * continuacion

print("Numero de variables:", numero_estados_generador)
print("Numero de restricciones:", numero_estados_generador * numero_acciones_generador)

Numero de variables: 4
Numero de restricciones: 8


In [6]:
inicio_tiempo = time.perf_counter()
problema_generador.solve(pulp.PULP_CBC_CMD(msg=0))
tiempo_lp_generador = (time.perf_counter() - inicio_tiempo) * 1000

print("Estado del solver:", pulp.LpStatus[problema_generador.status])
print("Tiempo observado:", round(tiempo_lp_generador, 3), "ms")

if pulp.LpStatus[problema_generador.status] != "Optimal":
    raise RuntimeError("El solver LP no encontro una solucion optima.")

valor_lp_generador = np.zeros(numero_estados_generador)
for estado_actual in range(numero_estados_generador):
    valor_lp_generador[estado_actual] = pulp.value(V_generador[estado_actual])

# Recuperar la politica: recalcular el valor de cada accion con V* ya fijo
valores_lp_por_accion = np.zeros((numero_estados_generador, numero_acciones_generador))

for estado_actual in range(numero_estados_generador):
    for accion_actual in range(numero_acciones_generador):
        continuacion = 0.0
        for estado_siguiente in range(numero_estados_generador):
            continuacion = continuacion + matrices_generador[accion_actual][estado_actual, estado_siguiente] * valor_lp_generador[estado_siguiente]
        valores_lp_por_accion[estado_actual, accion_actual] = retornos_generador[estado_actual, accion_actual] + gamma_generador * continuacion

politica_lp_generador = np.zeros(numero_estados_generador, dtype=int)
for estado_actual in range(numero_estados_generador):
    politica_lp_generador[estado_actual] = np.argmax(valores_lp_por_accion[estado_actual, :])

print()
print("Valor optimo (LP):")
for estado_actual in range(numero_estados_generador):
    print(" ", estados_generador[estado_actual], ":", round(float(valor_lp_generador[estado_actual]), 4))
print()
print("Politica obtenida con LP:")
for estado_actual in range(numero_estados_generador):
    print(" ", estados_generador[estado_actual], "->", acciones_generador[politica_lp_generador[estado_actual]])

Estado del solver: Optimal
Tiempo observado: 560.393 ms

Valor optimo (LP):
  Nuevo : 864.6747
  Bueno : 764.4337
  Regular : 744.6747
  Malo : 744.6747

Politica obtenida con LP:
  Nuevo -> Mantener
  Bueno -> Mantener
  Regular -> Reemplazar
  Malo -> Reemplazar


## 1.3 El LP reproduce la referencia

$V^*$ es único sin importar el método: si el LP está bien planteado, tiene que coincidir con el valor de referencia que ya obtuvimos con `jmarkov` en el paso 1.1.

In [7]:
print("V con LP:              ", np.round(valor_lp_generador, 4))
print("V de referencia (jmarkov):", np.round(np.array(valor_referencia_generador, dtype=float), 4))
print("¿Coinciden?               ", np.allclose(valor_lp_generador, valor_referencia_generador, atol=1e-3))

V con LP:               [864.6747 764.4337 744.6747 744.6747]
V de referencia (jmarkov): [864.6747 764.4337 744.6747 744.6747]
¿Coinciden?                True


---
# Ejercicio 2 — Inventario estacionario

Mismo ejemplo del cuaderno de la Semana 8: $X_t\in\{0,\ldots,4\}$, $a_t\in\{0,\ldots,4\}$, $X_t+a_t\leq 4$, demanda $P(D{=}0,1,2,3)=(0.10,0.30,0.40,0.20)$, $p=50$, $c=20$, $h=5$, $b=25$, $\gamma=0.95$.

In [8]:
capacidad_inventario = 4
precio_venta = 50
costo_pedido = 20
costo_almacenamiento = 5
costo_escasez = 25
penalizacion_infactible = -100000.0

valores_demanda = np.array([0, 1, 2, 3])
probabilidades_demanda = np.array([0.10, 0.30, 0.40, 0.20])

estados_inventario = np.array(["0", "1", "2", "3", "4"])
acciones_inventario = np.array(["0", "1", "2", "3", "4"])  # cantidad pedida, como etiqueta

numero_estados_inventario = len(estados_inventario)
numero_acciones_inventario = len(acciones_inventario)

gamma_inventario = 0.95


def construir_retornos_inventario(costo_escasez_modelo):

    retornos = np.zeros((numero_estados_inventario, numero_acciones_inventario))

    for inventario_actual in range(numero_estados_inventario):
        for cantidad_pedida in range(numero_acciones_inventario):

            inventario_disponible = inventario_actual + cantidad_pedida

            if inventario_disponible > capacidad_inventario:
                retornos[inventario_actual, cantidad_pedida] = penalizacion_infactible
            else:
                retorno_esperado = -costo_pedido * cantidad_pedida

                for indice_demanda in range(len(valores_demanda)):
                    demanda = valores_demanda[indice_demanda]
                    probabilidad = probabilidades_demanda[indice_demanda]

                    ventas = min(inventario_disponible, demanda)
                    inventario_final = max(0, inventario_disponible - demanda)
                    demanda_no_atendida = max(0, demanda - inventario_disponible)

                    retorno_demanda = precio_venta * ventas - costo_almacenamiento * inventario_final - costo_escasez_modelo * demanda_no_atendida
                    retorno_esperado = retorno_esperado + probabilidad * retorno_demanda

                retornos[inventario_actual, cantidad_pedida] = retorno_esperado

    return retornos


def construir_transiciones_inventario():

    matrices_por_accion = []

    for cantidad_pedida in range(numero_acciones_inventario):

        matriz_accion = np.zeros((numero_estados_inventario, numero_estados_inventario))

        for inventario_actual in range(numero_estados_inventario):
            inventario_disponible = inventario_actual + cantidad_pedida

            if inventario_disponible > capacidad_inventario:
                matriz_accion[inventario_actual, inventario_actual] = 1.0
            else:
                for indice_demanda in range(len(valores_demanda)):
                    demanda = valores_demanda[indice_demanda]
                    probabilidad = probabilidades_demanda[indice_demanda]
                    inventario_siguiente = max(0, inventario_disponible - demanda)
                    matriz_accion[inventario_actual, inventario_siguiente] = matriz_accion[inventario_actual, inventario_siguiente] + probabilidad

        matrices_por_accion.append(matriz_accion)

    return matrices_por_accion


retornos_inventario = construir_retornos_inventario(costo_escasez)
matrices_inventario = construir_transiciones_inventario()

print("Matriz de retornos:")
print(np.round(retornos_inventario, 2))

Matriz de retornos:
[[    -42.5       4.5      27.5      18.5      -6.5]
 [     24.5      47.5      38.5      13.5 -100000. ]
 [     67.5      58.5      33.5 -100000.  -100000. ]
 [     78.5      53.5 -100000.  -100000.  -100000. ]
 [     73.5 -100000.  -100000.  -100000.  -100000. ]]


### Primero, la referencia con `jmarkov`

In [9]:
matrices_jmarkov_inventario = {}
for cantidad_pedida in range(numero_acciones_inventario):
    matrices_jmarkov_inventario[acciones_inventario[cantidad_pedida]] = matrices_inventario[cantidad_pedida]

mdp_inventario = dtmdp(
    estados_inventario, acciones_inventario, matrices_jmarkov_inventario, retornos_inventario, gamma_inventario
)

valor_referencia_inventario, politica_referencia_inventario = mdp_inventario.solve(1e-10, minimize=False)

print("Valor de referencia (jmarkov):")
for estado_actual in range(numero_estados_inventario):
    print(" ", estados_inventario[estado_actual], ":", round(float(valor_referencia_inventario[estado_actual]), 4))
print()
print("Politica de referencia (jmarkov):", politica_referencia_inventario)

Valor de referencia (jmarkov):
  0 : 864.0
  1 : 884.0
  2 : 904.0
  3 : 924.0
  4 : 937.3702

Politica de referencia (jmarkov): {np.str_('0'): np.str_('3'), np.str_('1'): np.str_('2'), np.str_('2'): np.str_('1'), np.str_('3'): np.str_('0'), np.str_('4'): np.str_('0')}


### Ahora, con LP (`PuLP`)

Igual que en el Ejercicio 1: una variable $V_s$ por estado, función objetivo $\min\sum_s V_s$, y una restricción por cada par (estado, cantidad pedida).

In [10]:
problema_inventario = pulp.LpProblem("MDP_inventario", pulp.LpMinimize)

V_inventario = []
for estado_actual in range(numero_estados_inventario):
    V_inventario.append(pulp.LpVariable(f"V_{estado_actual}", lowBound=None))

problema_inventario += pulp.lpSum(V_inventario)

for estado_actual in range(numero_estados_inventario):
    for accion_actual in range(numero_acciones_inventario):

        continuacion = pulp.lpSum(
            matrices_inventario[accion_actual][estado_actual, estado_siguiente] * V_inventario[estado_siguiente]
            for estado_siguiente in range(numero_estados_inventario)
        )

        problema_inventario += V_inventario[estado_actual] >= retornos_inventario[estado_actual, accion_actual] + gamma_inventario * continuacion

problema_inventario.solve(pulp.PULP_CBC_CMD(msg=0))
estado_lp_inventario = pulp.LpStatus[problema_inventario.status]

if estado_lp_inventario != "Optimal":
    raise RuntimeError("El solver LP no encontro una solucion optima.")

valor_lp_inventario = np.zeros(numero_estados_inventario)
for estado_actual in range(numero_estados_inventario):
    valor_lp_inventario[estado_actual] = pulp.value(V_inventario[estado_actual])

# Recuperar la politica: recalcular el valor de cada accion con V* ya fijo
valores_lp_inventario_por_accion = np.zeros((numero_estados_inventario, numero_acciones_inventario))

for estado_actual in range(numero_estados_inventario):
    for accion_actual in range(numero_acciones_inventario):
        continuacion = 0.0
        for estado_siguiente in range(numero_estados_inventario):
            continuacion = continuacion + matrices_inventario[accion_actual][estado_actual, estado_siguiente] * valor_lp_inventario[estado_siguiente]
        valores_lp_inventario_por_accion[estado_actual, accion_actual] = retornos_inventario[estado_actual, accion_actual] + gamma_inventario * continuacion

politica_lp_inventario = np.zeros(numero_estados_inventario, dtype=int)
for estado_actual in range(numero_estados_inventario):
    politica_lp_inventario[estado_actual] = np.argmax(valores_lp_inventario_por_accion[estado_actual, :])

restricciones_lp_inventario = numero_estados_inventario * numero_acciones_inventario

print("Estado del solver:", estado_lp_inventario)
print("Numero de restricciones:", restricciones_lp_inventario, "( =", numero_estados_inventario, "x", numero_acciones_inventario, ")")
print()

for estado_actual in range(numero_estados_inventario):
    print(
        "Estado", estados_inventario[estado_actual],
        "-> V:", round(float(valor_lp_inventario[estado_actual]), 4),
        "- pedir", politica_lp_inventario[estado_actual]
    )

print()
print("¿Coincide con la referencia de jmarkov?", np.allclose(valor_lp_inventario, valor_referencia_inventario, atol=1e-3))

Estado del solver: Optimal
Numero de restricciones: 25 ( = 5 x 5 )

Estado 0 -> V: 864.0 - pedir 3
Estado 1 -> V: 884.0 - pedir 2
Estado 2 -> V: 904.0 - pedir 1
Estado 3 -> V: 924.0 - pedir 0
Estado 4 -> V: 937.3702 - pedir 0

¿Coincide con la referencia de jmarkov? True


### Política estacionaria $(s^*,S^*)$

In [11]:
punto_reorden = -1
nivel_objetivo = -1
estructura_sS = True

print("Politica optima (LP):")

for inventario_actual in range(numero_estados_inventario):

    cantidad_pedida = politica_lp_inventario[inventario_actual]
    inventario_despues_pedido = inventario_actual + cantidad_pedida

    print("Inventario", inventario_actual, "-> pedir", cantidad_pedida, "-> nivel despues del pedido:", inventario_despues_pedido)

    if cantidad_pedida > 0:
        punto_reorden = inventario_actual
        if nivel_objetivo == -1:
            nivel_objetivo = inventario_despues_pedido
        elif inventario_despues_pedido != nivel_objetivo:
            estructura_sS = False

print("\n¿La politica tiene estructura (s*, S*)?", estructura_sS)
if estructura_sS and punto_reorden >= 0:
    print("s* =", punto_reorden)
    print("S* =", nivel_objetivo)

Politica optima (LP):
Inventario 0 -> pedir 3 -> nivel despues del pedido: 3
Inventario 1 -> pedir 2 -> nivel despues del pedido: 3
Inventario 2 -> pedir 1 -> nivel despues del pedido: 3
Inventario 3 -> pedir 0 -> nivel despues del pedido: 3
Inventario 4 -> pedir 0 -> nivel despues del pedido: 4

¿La politica tiene estructura (s*, S*)? True
s* = 2
S* = 3


---
# Ejercicio abierto — Gestión de flota de vehículos

Mismo ejemplo del cuaderno de la Semana 8.

$$
\mathcal S=\{\text{Óptimo},\text{Funcional},\text{Desgastado},\text{Crítico}\}, \qquad \mathcal A=\{\text{Mantener},\text{Reacondicionar}\}.
$$

$$
P^{(M)}=\begin{pmatrix}0.80&0.20&0&0\\0&0.75&0.25&0\\0&0&0.65&0.35\\0&0&0&1\end{pmatrix}, \qquad P^{(R)}=\begin{pmatrix}1&0&0&0\\0.80&0.20&0&0\\0.80&0.20&0&0\\0.80&0.20&0&0\end{pmatrix}.
$$

| Estado | Mantener | Reacondicionar |
|:---|---:|---:|
| Óptimo | 200 | -10000 |
| Funcional | 160 | -80 |
| Desgastado | 100 | -80 |
| Crítico | 40 | -80 |

Use $\gamma=0.92$.

### Actividades

1. Construya `P` como una lista de dos matrices y `R` como una matriz bidimensional (puede reutilizar las que ya construyó en el cuaderno de la Semana 8).
2. Obtenga primero la referencia con `jmarkov` (`dtmdp.solve`).
3. Formule y resuelva el mismo problema como LP con `PuLP`, directamente (variables `V`, objetivo `pulp.lpSum(V)`, restricciones agregadas con `+=`), igual que en los Ejercicios 1 y 2.
4. Identifique desde qué estado conviene reacondicionar.
5. Calcule el número de restricciones del LP: $|\mathcal S||\mathcal A|$.
6. Para 20 estados y 2 acciones, determine el número de restricciones del LP. Explique por qué ese número **sí** se puede calcular de antemano solo con $|\mathcal S|$ y $|\mathcal A|$, a diferencia del número de iteraciones de VI (que no se puede predecir solo a partir de $|\mathcal S|$, como vio en la Semana 8).

## Uso crítico de IA

Antes de aceptar el código generado, compruebe:

$$
R[\text{Funcional},\text{Reacondicionar}] = 200-280 = -80.
$$

También verifique que:

- todas las filas de las matrices de transición sumen 1;
- la primera fila de Reacondicionar sea `[1, 0, 0, 0]`;
- el código obtiene primero la referencia con `dtmdp` y luego formula y resuelve el LP con `PuLP` — no al revés;
- las variables de `PuLP` se crean **sin cota inferior** (`lowBound=None`) y las restricciones se agregan directamente con `+=`, sin envolver el procedimiento en una función aparte.

### Prompt sugerido

```text
Estoy resolviendo el mismo MDP de horizonte infinito del cuaderno anterior,
pero ahora con Programacion Lineal usando PuLP.

Ayudame a construir los arreglos del ejercicio de flota (los mismos P y R que
ya use en el cuaderno de VI y PI). Usa solamente ciclos for, condicionales y
arreglos de NumPy para construir P y R.

Primero arma un dtmdp con esos arreglos y resuelvelo, para tener el valor de
referencia. Despues formula el mismo problema como programa lineal usando
PuLP directamente (como hicimos en los Ejercicios 1 y 2: variables V con
lowBound=None, objetivo pulp.lpSum(V), restricciones agregadas una por una
con +=) y confirma que da el mismo valor. No lo envuelvas en una funcion.

Comprueba que R[Funcional, Reacondicionar] sea -80 y que todas las filas de
las matrices de transicion sumen 1.

Para el caso de 20 estados y 2 acciones, calcula el numero de restricciones
del LP y explica por que ese numero si se puede determinar de antemano,
a diferencia del numero de iteraciones de VI.
```

In [12]:
# Desarrolle aqui el ejercicio abierto.
# Construya los estados, las acciones, las dos matrices de transicion
# y la matriz de retornos (puede reutilizarlas del cuaderno de la Semana 8).
# Obtenga primero la referencia con dtmdp, luego formule y resuelva el LP con PuLP directamente.

---
<small>Universidad de los Andes · Departamento de Ingeniería Industrial · Modelos de Decisión en el Tiempo</small>